# 04 — Prepare CNN Windows
# Giai đoạn 1 — Mục 1.5 — Chuẩn bị sliding windows cho CNN raw và envelope
# 
 **Đầu ra**:
- `outputs/tables/windows_cnn_raw.parquet`
- `outputs/tables/windows_cnn_env.parquet`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from scipy.signal import resample_poly
from common import io_utils, dsp, features, config as cfg

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

- 1. Đọc manifest ĐÃ LÀM SẠCH và cấu hình bandpass

In [3]:
manifest_clean = pd.read_csv(TABLES_DIR / "manifest_clean.csv")

with open(TABLES_DIR / "bandpass_config.json") as f:
    bandpass_cfg = json.load(f)
BAND_HZ = tuple(bandpass_cfg["band_hz"])
LP_CUTOFF_HZ = bandpass_cfg["lp_cutoff_hz"]

WINDOW_SIZE_RAW = 2048
STRIDE_RAW = 1024
WINDOW_SIZE_ENV = 1024
STRIDE_ENV = 512
TARGET_FS = 12000

- Prepare CNN Windows

In [4]:
all_windows_raw = []
all_windows_env = []

for _, row in manifest_clean.iterrows():
    if row['label'] is None or pd.isna(row['label']):
        continue
    
    # Load signal và resample dựa trên tần số lấy mẫu thực tế (row['fs'])
    x = io_utils.load_de_signal(Path(row['file_path']))
    current_fs = row['fs']
    if current_fs != TARGET_FS:
        x = resample_poly(x, TARGET_FS, current_fs)
        
    file_id = f"{row['label']}_{row['load_hp']}_{row.get('fault_diameter_mils', '')}_{Path(row['file_path']).name}"
    
    # Xử lý Raw
    w_raw = features.make_sliding_windows(x, WINDOW_SIZE_RAW, overlap_ratio=0.5, file_id=file_id, label=row['label'])
    all_windows_raw.append(w_raw)
    
    # Xử lý Envelope
    envelope = dsp.square_law_envelope(x, fs=TARGET_FS, band=BAND_HZ, lp_cutoff=LP_CUTOFF_HZ)
    w_env = features.make_sliding_windows(envelope, WINDOW_SIZE_ENV, overlap_ratio=0.5, file_id=file_id, label=row['label'])
    all_windows_env.append(w_env)

- 3. Kết xuất dữ liệu

In [5]:
windows_raw_df = pd.concat(all_windows_raw, ignore_index=True)
windows_env_df = pd.concat(all_windows_env, ignore_index=True)

windows_raw_df.to_parquet(TABLES_DIR / "windows_cnn_raw.parquet")
windows_env_df.to_parquet(TABLES_DIR / "windows_cnn_env.parquet")

print(f"Hoàn tất! Raw windows: {len(windows_raw_df)}, Envelope windows: {len(windows_env_df)}")

Hoàn tất! Raw windows: 4703, Envelope windows: 9465
